In [1]:
import json
import os
from collections import defaultdict

BASE_DIR       = r"D:\Memoire\mooccubeX"
TRANSLATED_DIR = os.path.join(BASE_DIR, "translated")

# ── 1. Charger TOUS les IDs de cours ──────────────────────────────────────
course_ids_str = set()   # ex: {'C_584313', ...}
course_ids_num = set()   # ex: {'584313', ...}  (sans préfixe)

with open(os.path.join(TRANSLATED_DIR, "course_translated.json"), 'r',
          encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                cid = str(obj.get("id", ""))
                course_ids_str.add(cid)
                # Extraire la partie numérique (après 'C_')
                if cid.startswith("C_"):
                    course_ids_num.add(cid[2:])  # '584313'
            except:
                pass

print(f"✅ Cours chargés  : {len(course_ids_str):,}")
print(f"   Exemples IDs   : {list(course_ids_str)[:5]}")
print(f"   Exemples num.  : {list(course_ids_num)[:5]}")

✅ Cours chargés  : 3,781
   Exemples IDs   : ['C_676635', 'C_1789682', 'C_1904003', 'C_735064', 'C_682659']
   Exemples num.  : ['629522', '685689', '2308096', '681353', '1741508']


In [2]:
# ── 2. Analyser course_order dans user.json ────────────────────────────────
sample_orders = []   # liste de course_order extraits
n_users_with_orders = 0

with open(os.path.join(BASE_DIR, "user.json"), 'r',
          encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 20000:   # échantillon de 20k users
            break
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            orders = obj.get("course_order", [])
            if orders:
                n_users_with_orders += 1
                sample_orders.extend(orders[:3])  # max 3 par user
        except:
            pass

print(f"✅ Users scannés          : 20,000")
print(f"   Users avec course_order : {n_users_with_orders:,}")
print(f"   Échantillon d'orders    : {sample_orders[:10]}")
print(f"   Type des valeurs        : {type(sample_orders[0]) if sample_orders else 'N/A'}")

✅ Users scannés          : 20,000
   Users avec course_order : 20,000
   Échantillon d'orders    : [682129, 2294668, 597214, 605512, 597211, 1903985, 696679, 1704639, 943255, 682442]
   Type des valeurs        : <class 'int'>


In [3]:
# ── 3. Tester les 3 hypothèses de correspondance ──────────────────────────
test_orders = sample_orders[:500]

# Hypothèse A : int → 'C_' + str(int)
match_A = sum(1 for o in test_orders if f"C_{o}" in course_ids_str)

# Hypothèse B : int → str(int) dans les IDs numériques
match_B = sum(1 for o in test_orders if str(o) in course_ids_num)

# Hypothèse C : les valeurs sont déjà des strings type 'C_XXXX'
match_C = sum(1 for o in test_orders if str(o) in course_ids_str)

total = len(test_orders)
print(f"📊 Test sur {total} valeurs de course_order :")
print(f"   Hypothèse A (C_{{int}})  : {match_A}/{total} = {100*match_A//total}%")
print(f"   Hypothèse B (num only) : {match_B}/{total} = {100*match_B//total}%")
print(f"   Hypothèse C (direct)   : {match_C}/{total} = {100*match_C//total}%")

# Afficher quelques exemples pour mieux comprendre
print("\n📄 Exemples :")
for o in test_orders[:5]:
    a = f"C_{o}" in course_ids_str
    b = str(o) in course_ids_num
    print(f"   order={o}  →  C_{o} exists={a}  |  '{o}' in num={b}")

📊 Test sur 500 valeurs de course_order :
   Hypothèse A (C_{int})  : 431/500 = 86%
   Hypothèse B (num only) : 431/500 = 86%
   Hypothèse C (direct)   : 0/500 = 0%

📄 Exemples :
   order=682129  →  C_682129 exists=True  |  '682129' in num=True
   order=2294668  →  C_2294668 exists=False  |  '2294668' in num=False
   order=597214  →  C_597214 exists=False  |  '597214' in num=False
   order=605512  →  C_605512 exists=False  |  '605512' in num=False
   order=597211  →  C_597211 exists=False  |  '597211' in num=False


In [4]:
# ── 4. Si aucune hypothèse directe ne fonctionne : chercher dans course.json original
# (au cas où l'ID dans course_order est différent du champ 'id')

print("🔎 Inspection du fichier course.json ORIGINAL (non traduit)...")
original_course_path = os.path.join(BASE_DIR, "course.json")

if not os.path.exists(original_course_path):
    print("   ❌ Fichier course.json original introuvable dans BASE_DIR")
    print(f"   Chemin testé : {original_course_path}")
else:
    orig_samples = []
    with open(original_course_path, 'r', encoding='utf-8', errors='replace') as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            line = line.strip()
            if line:
                try:
                    orig_samples.append(json.loads(line))
                except:
                    pass
    if orig_samples:
        print(f"   Clés dans course.json original : {list(orig_samples[0].keys())}")
        print(f"   Premier ID : {orig_samples[0].get('id')}")

🔎 Inspection du fichier course.json ORIGINAL (non traduit)...
   Clés dans course.json original : ['id', 'name', 'field', 'prerequisites', 'about', 'resource']
   Premier ID : C_584313


In [5]:
# ── 5. Vérifier la relation course_order via user-video (chemin alternatif) ──
# Si course_order ne matche pas → on construira User→Course
# via : User → Video → CCID → Concept → Course

print("🔄 Vérification du chemin alternatif User→Course via user-video...")

# Extraire un échantillon d'user_ids depuis user-video.json
uv_user_ids = set()
with open(os.path.join(BASE_DIR, "user-video.json"), 'r',
          encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            uid = obj.get("user_id")
            if uid:
                uv_user_ids.add(uid)
        except:
            pass

print(f"   Users distincts dans user-video (5k lignes) : {len(uv_user_ids)}")
print(f"   Exemples : {list(uv_user_ids)[:5]}")

# Vérifier la structure d'une entrée user-video
with open(os.path.join(BASE_DIR, "user-video.json"), 'r',
          encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                print(f"\n   Clés : {list(obj.keys())}")
                seq = obj.get('seq', [])
                print(f"   Nombre de vidéos regardées : {len(seq)}")
                print(f"   Exemple video_id : {seq[0].get('video_id') if seq else 'N/A'}")
                break
            except:
                pass

🔄 Vérification du chemin alternatif User→Course via user-video...
   Users distincts dans user-video (5k lignes) : 5000
   Exemples : ['U_525645', 'U_436318', 'U_561884', 'U_947382', 'U_331911']

   Clés : ['seq', 'user_id']
   Nombre de vidéos regardées : 6
   Exemple video_id : V_1395633


In [6]:
# ── 6. Vérifier video_id-ccid.txt : est-ce que les V_ids matchent video_translated ?
print("🔗 Vérification video_id-ccid.txt vs video_translated.json...")

# Charger un échantillon de ccids depuis video_translated
video_ccids = set()
with open(os.path.join(TRANSLATED_DIR, "video_translated.json"), 'r',
          encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 10000:
            break
        line = line.strip()
        if line:
            try:
                obj = json.loads(line)
                ccid = obj.get("ccid")
                if ccid:
                    video_ccids.add(ccid)
            except:
                pass

print(f"   CCIDs dans video_translated (10k) : {len(video_ccids)}")

# Tester correspondance avec video_id-ccid.txt
matched_ccid = 0
total_vccid  = 0
vids_sample  = []

with open(os.path.join(BASE_DIR, "video_id-ccid.txt"), 'r',
          encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 10000:
            break
        parts = line.strip().split('\t')
        if len(parts) == 2:
            total_vccid += 1
            vid, ccid = parts
            if ccid in video_ccids:
                matched_ccid += 1
            if i < 3:
                vids_sample.append((vid, ccid))

print(f"   Match V_id→ccid dans video_translated : {matched_ccid}/{total_vccid} ({100*matched_ccid//total_vccid if total_vccid else 0}%)")
print(f"   Exemples : {vids_sample}")

🔗 Vérification video_id-ccid.txt vs video_translated.json...
   CCIDs dans video_translated (10k) : 10000
   Match V_id→ccid dans video_translated : 5042/10000 (50%)
   Exemples : [('V_234845', '0000363DB5B6E0869C33DC5901307461'), ('V_234876', '0000363DB5B6E0869C33DC5901307461'), ('V_234907', '0000363DB5B6E0869C33DC5901307461')]
